In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
import time
import random

# ---- Paths ----
file_path = '/content/drive/MyDrive/Major/olist_order_dataset_cleaned.xlsx'

# ---- Parameters ----
iterations = 2
rows_per_iteration = 50
interval = 5  # seconds

# ---- Step 1: Load the base dataset once ----
df = pd.read_excel(file_path)
print(f"✅ Loaded {len(df)} existing records.")

# ---- Step 2: Determine the current max order number ----
df['order_id_num'] = df['order_id'].str.extract('(\d+)').astype(float)
current_count = int(df['order_id_num'].max() if not df['order_id_num'].isna().all() else 1000)
df.drop(columns=['order_id_num'], inplace=True)

# ---- Helper function to generate sequential unique IDs ----
def generate_unique_ids(start, n, prefix="ORDER"):
    return [f"{prefix}{i:06d}" for i in range(start + 1, start + n + 1)]

# ---- Step 3: Start data ingestion loop ----
print("🚀 Starting controlled ingestion...")

for i in range(iterations):
    try:
        # Pick random sample to duplicate
        sample_rows = df.sample(
            n=rows_per_iteration,
            replace=True,
            random_state=random.randint(0, 9999)
        ).copy()

        # Assign new unique order IDs
        new_ids = generate_unique_ids(current_count, rows_per_iteration)
        sample_rows['order_id'] = new_ids
        current_count += rows_per_iteration

        # Generate unique customer IDs
        sample_rows['customer_id'] = [
            f"CUST{current_count + j:06d}" for j in range(rows_per_iteration)
        ]

        # Append to the master DataFrame
        df = pd.concat([df, sample_rows], ignore_index=True)

        # ---------------------------------------------------------
        # LIGHT POST-INGESTION CLEANING (SAFE & NON-DESTRUCTIVE)
        # ---------------------------------------------------------

        # Convert date columns
        date_cols = [
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "order_purchase_timestamp_date",
            "order_delivered_customer_dateDate",
            "order_estimated_delivery_dateDate"
        ]
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")

        # Convert numeric columns
        num_cols = [
            "order_item_id",
            "price",
            "freight_value",
            "payment_installments",
            "payment_value",
            "review_score",
            "shipping_days",
            "delivery_delay_days",
            "order_year"
        ]
        for col in num_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # Clean string columns
        string_cols = [
            "order_id",
            "customer_id",
            "order_status",
            "customer_city",
            "customer_state",
            "payment_type",
            "product_category_name_english",
            "general_category",
            "order_month_name"
        ]
        for col in string_cols:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip()

        # ---------------------------------------------------------

        # Save after each iteration
        df.to_excel(file_path, index=False)
        print(f"✅ Iteration {i+1}/{iterations}: Added {rows_per_iteration} rows → Cleaned → Saved.")
        print(f"⏰ Waiting {interval} seconds...\n")

        time.sleep(interval)

    except Exception as e:
        print(f"❌ Error in iteration {i+1}: {e}")

print("🎉 Data ingestion + light cleaning complete! Everything saved to Drive.")


<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:18: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-1592425861.py:18: SyntaxWarning: invalid escape sequence '\d'
  df['order_id_num'] = df['order_id'].str.extract('(\d+)').astype(float)


✅ Loaded 130040 existing records.
🚀 Starting controlled ingestion...
✅ Iteration 1/2: Added 50 rows → Cleaned → Saved.
⏰ Waiting 5 seconds...

✅ Iteration 2/2: Added 50 rows → Cleaned → Saved.
⏰ Waiting 5 seconds...

🎉 Data ingestion + light cleaning complete! Everything saved to Drive.


In [6]:
!pip install pmdarima

!pip install reportlab

In [7]:
# full_quickstats_arima_combined_with_arima_details_email_short_final.py
# Combines QuickStats alerts + Auto-ARIMA forecasting
# PDF report contains both alerts and ARIMA graph + ARIMA findings (findings below graph)
# Email contains greeting, one alert line, and the ARIMA summary sentence

import os
import pandas as pd
import smtplib
import traceback
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
import matplotlib.pyplot as plt
import numpy as np

# PDF generation imports
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont

# ARIMA import
from pmdarima import auto_arima

# ----------------- CONFIG -----------------
DATA_PATH = "/content/drive/MyDrive/Major/olist_order_dataset_cleaned.xlsx"
ALERTS_PATH = "/content/drive/MyDrive/Major/Alerts_Data.xlsx"
PDF_REPORT_PATH = "/content/drive/MyDrive/Major/QuickStats_ARIMA_Report.pdf"
ARIMA_PLOT_PATH = "/content/drive/MyDrive/Major/QuickStats_ARIMA_Plot.png"

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

SENDERS = [
    {"email": "aurevella234@gmail.com", "password": "etkk dmew ocmw tdvd"}
]

RECEIVER_EMAILS = ["sapna.bhola86@gmail.com"]

CATEGORY_COL = "general_category"
PRICE_COL = "payment_value"
REVIEW_COL = "review_score"
DELAY_COL = "delivery_delay_days"

BRL_TO_INR = 16.5

# ----------------- UTILS -----------------
def safe_read_excel(path):
    try:
        return pd.read_excel(path)
    except Exception as e:
        print(f"❌ Error reading {path}: {e}")
        return pd.DataFrame()

def read_alerts_sheets(path):
    if not os.path.exists(path):
        return pd.DataFrame(), pd.DataFrame()
    try:
        all_sheets = pd.read_excel(path, sheet_name=None)
        return all_sheets.get("log", pd.DataFrame()), all_sheets.get("snapshot", pd.DataFrame())
    except Exception as e:
        print(f"❌ Failed to read sheets from {path}: {e}")
        return pd.DataFrame(), pd.DataFrame()

def write_alerts_sheets(path, log_df, snapshot_df):
    try:
        with pd.ExcelWriter(path, engine="openpyxl", mode="w") as writer:
            log_df.to_excel(writer, sheet_name="log", index=False)
            snapshot_df.to_excel(writer, sheet_name="snapshot", index=False)
        print("✅ Alerts file updated with 'log' and 'snapshot' sheets:", path)
    except Exception as e:
        print("❌ Failed to write alerts sheets:", e)
        traceback.print_exc()

def format_brl_inr(value_brl):
    try:
        value_inr = value_brl * BRL_TO_INR
    except Exception:
        value_inr = 0
    return f"R${value_brl:,.2f} (≈ ₹{value_inr:,.0f})"

def format_delay(value):
    return "No delay (early deliveries)" if value < 0 else f"{value:.1f} days"

# ----------------- EMAIL -----------------
def send_email(subject, body, attachment_path=None):
    sender = SENDERS[0]
    try:
        msg = MIMEMultipart()
        msg["From"] = sender["email"]
        msg["To"] = ", ".join(RECEIVER_EMAILS)
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        if attachment_path and os.path.exists(attachment_path):
            with open(attachment_path, "rb") as f:
                part = MIMEApplication(f.read(), Name=os.path.basename(attachment_path))
            part["Content-Disposition"] = f'attachment; filename="{os.path.basename(attachment_path)}"'
            msg.attach(part)

        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.starttls()
            server.login(sender["email"], sender["password"])
            server.send_message(msg)
        print(f"📧 Email alert sent successfully from {sender['email']}")
    except Exception as e:
        print(f"❌ Failed to send email from {sender['email']}: {e}")
        traceback.print_exc()

# ----------------- PDF GENERATION -----------------
def generate_pdf_report(pdf_path, plot_path, timestamp, alerts, current_top_5, top_category,
                        avg_review, total_sales_fmt, avg_delay_fmt, comparison_data, new_entries, dropped,
                        arima_order=None, forecast_df=None):
    try:
        pdfmetrics.registerFont(UnicodeCIDFont("HeiseiMin-W3"))
    except Exception:
        pass

    doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=40, leftMargin=40, topMargin=60, bottomMargin=40)
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "TitleStyle", parent=styles["Title"], alignment=TA_CENTER,
        fontName="HeiseiMin-W3", fontSize=18, textColor=colors.HexColor("#1F4E79"), spaceAfter=20
    )
    section_header = ParagraphStyle(
        "SectionHeader", parent=styles["Heading2"],
        fontName="HeiseiMin-W3", fontSize=12, textColor=colors.HexColor("#0A2540"),
        spaceBefore=10, spaceAfter=6
    )
    normal_text = ParagraphStyle("NormalText", parent=styles["Normal"], fontName="HeiseiMin-W3", fontSize=10, leading=13)
    footer_style = ParagraphStyle("Footer", parent=styles["Normal"], alignment=TA_CENTER,
                                  fontName="HeiseiMin-W3", fontSize=9, textColor=colors.grey)

    content = []
    content.append(Paragraph("QuickStats Performance Summary Report", title_style))
    content.append(Paragraph(f"<b>Date:</b> {timestamp}", normal_text))
    content.append(Spacer(1, 10))

    # Alerts
    content.append(Paragraph("📢 Alert Summary", section_header))
    for alert in alerts:
        content.append(Paragraph(f"• {alert}", normal_text))
    content.append(Spacer(1, 10))

    # Metrics Table
    table_data = [
        ["Top 5 Categories", Paragraph(", ".join(current_top_5), normal_text)],
        ["Top Category by Sales", Paragraph(top_category, normal_text)],
        ["Average Review Score", Paragraph(str(avg_review), normal_text)],
        ["Total Sales", Paragraph(total_sales_fmt, normal_text)],
        ["Average Delivery Delay", Paragraph(avg_delay_fmt, normal_text)]
    ]
    table = Table(table_data, colWidths=[160, 340])
    table.setStyle(TableStyle([
        ("FONTNAME", (0, 0), (-1, -1), "HeiseiMin-W3"),
        ("ALIGN", (0, 0), (-1, -1), "LEFT"),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("INNERGRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("BOX", (0, 0), (-1, -1), 0.25, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.HexColor("#F7F9FB")])
    ]))
    content.append(table)
    content.append(Spacer(1, 12))

    # Category Comparison
    content.append(Paragraph("📈 Category Sales Comparison (vs Previous Report)", section_header))
    if new_entries:
        content.append(Paragraph(f"🆕 New categories entered Top 5: {', '.join(new_entries)}", normal_text))
    if dropped:
        content.append(Paragraph(f"⬇ Categories dropped from Top 5: {', '.join(dropped)}", normal_text))
    content.append(Spacer(1, 6))
    if comparison_data:
        comp_table_data = [["Category", "Previous Sales (BRL)", "Added Sales (BRL)", "Current Sales (BRL)"]]
        for cat, prev_sales, added_sales, cur_sales in comparison_data:
            comp_table_data.append([cat, f"{prev_sales:,.2f}", f"{added_sales:,.2f}", f"{cur_sales:,.2f}"])
        t = Table(comp_table_data, colWidths=[180, 110, 110, 110])
        t.setStyle(TableStyle([
            ("FONTNAME", (0, 0), (-1, -1), "HeiseiMin-W3"),
            ("FONTSIZE", (0, 0), (-1, -1), 9),
            ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9E1F2")),
            ("ALIGN", (1, 1), (-1, -1), "RIGHT")
        ]))
        content.append(t)
    content.append(Spacer(1, 15))

    # ARIMA Graph and findings ON THE SAME PAGE (graph starts on new page)
    content.append(PageBreak())
    content.append(Paragraph("Forecast Graph", section_header))
    if plot_path and os.path.exists(plot_path):
        content.append(Image(plot_path, width=500, height=300))
    content.append(Spacer(1, 12))

    # Findings directly below the graph (same page)
    content.append(Paragraph("📊 Sales Forecast Findings", section_header))
    if arima_order and forecast_df is not None and not forecast_df.empty:
        last_actual = float(forecast_df['forecast_sales'].iloc[0])
        future_min = float(forecast_df['lower_bound'].min())
        future_max = float(forecast_df['upper_bound'].max())
        future_last = float(forecast_df['forecast_sales'].iloc[-1])
        trend = "positive and upward" if future_last > last_actual else "flat/slightly downward"

        findings_text = (
            f"Historical sales show strong growth in recent periods.\n"
            f"Expected future sales rise from {last_actual:,.0f} to {future_last:,.0f} within {len(forecast_df)} months.\n"
            f"The confidence interval suggests sales may vary between {future_min:,.0f} and {future_max:,.0f}, showing some uncertainty.\n"
            f"Overall forecast trend = {trend}."
        )
        for line in findings_text.split("\n"):
            content.append(Paragraph(line, normal_text))
    else:
        content.append(Paragraph("ARIMA forecast details are unavailable (not enough data).", normal_text))

    content.append(Spacer(1, 20))
    content.append(Paragraph("Confidential | Auto-generated by QuickStats Monitoring System", footer_style))

    doc.build(content)
    print(f"✅ PDF report generated with ARIMA graph and findings: {pdf_path}")

# ----------------- MAIN -----------------
def main():
    print("🔁 Loading data from:", DATA_PATH)
    df = safe_read_excel(DATA_PATH)
    if df.empty:
        print("❌ No data loaded. Aborting alert run.")
        return

    # Ensure numeric columns exist and are numeric
    df[PRICE_COL] = pd.to_numeric(df.get(PRICE_COL, pd.Series(dtype=float)), errors="coerce").fillna(0)
    df[REVIEW_COL] = pd.to_numeric(df.get(REVIEW_COL, pd.Series(dtype=float)), errors="coerce").fillna(0)
    df[DELAY_COL] = pd.to_numeric(df.get(DELAY_COL, pd.Series(dtype=float)), errors="coerce").fillna(0)

    # QuickStats Metrics
    category_sales = df.groupby(CATEGORY_COL)[PRICE_COL].sum().sort_values(ascending=False)
    current_top_5 = list(category_sales.head(5).index.astype(str))
    top_category = current_top_5[0] if current_top_5 else ""
    top_category_sales_brl = float(category_sales.iloc[0]) if not category_sales.empty else 0.0
    avg_review = round(df[REVIEW_COL].mean(), 2) if REVIEW_COL in df.columns else 0.0
    total_sales_brl = float(round(df[PRICE_COL].sum(), 2))
    avg_delay = round(df[DELAY_COL].mean(), 2) if DELAY_COL in df.columns else 0.0
    total_sales_fmt = format_brl_inr(total_sales_brl)
    avg_delay_fmt = format_delay(avg_delay)

    # Previous alerts loading
    log_df_prev, snapshot_prev = read_alerts_sheets(ALERTS_PATH)
    prev_top_5 = []
    prev_sales_map = {}
    if not snapshot_prev.empty and "Category" in snapshot_prev.columns and "Sales_BRL" in snapshot_prev.columns:
        prev_snapshot_sorted = snapshot_prev.sort_values("Sales_BRL", ascending=False)
        prev_top_5 = list(prev_snapshot_sorted.head(5)["Category"].astype(str))
        prev_sales_map = dict(zip(snapshot_prev["Category"].astype(str), snapshot_prev["Sales_BRL"].astype(float)))

    # Comparison & alerts
    all_categories = set(category_sales.index.astype(str)).union(set(prev_sales_map.keys()))
    comparison_data = []
    significant_changes = []
    # ---- FIXED BLOCK: compute raw_added (signed) and added (clamped to >=0) ----
    for cat in sorted(all_categories):
        cur = float(category_sales.get(cat, 0.0)) if cat in category_sales.index.astype(str) else 0.0
        prev = float(prev_sales_map.get(cat, 0.0))

        raw_added = cur - prev               # signed difference (can be negative)
        added = raw_added if raw_added > 0 else 0   # only positive increases show in "Added Sales"

        comparison_data.append((cat, prev, added, cur))

        # use raw_added to detect significant increases or decreases
        if abs(raw_added) >= 10.0:
            trend = "increased" if raw_added > 0 else "decreased"
            significant_changes.append(f"'{cat}' has {trend} by R${abs(raw_added):,.2f} (from R${prev:,.2f} to R${cur:,.2f}).")
    # ---------------------------------------------------------------------------

    new_entries = [c for c in current_top_5 if c not in prev_top_5]
    dropped = [c for c in prev_top_5 if c not in current_top_5]

    alerts = []
    prev_top_cat = prev_top_5[0] if prev_top_5 else None
    if prev_top_cat and prev_top_cat != top_category:
        alerts.append(f"Top category has changed from '{prev_top_cat}' to '{top_category}'.")
    elif top_category:
        alerts.append(f"Top category remains '{top_category}'.")
    prev_top_sales = prev_sales_map.get(top_category, None)
    if prev_top_sales is not None and prev_top_sales > 0:
        diff_top = top_category_sales_brl - prev_top_sales
        if abs(diff_top) >= 10.0:
            trend = "increased" if diff_top > 0 else "decreased"
            alerts.append(f"Sales for top category '{top_category}' have {trend} by R${abs(diff_top):,.2f} since last snapshot.")
    alerts.append(f"'{top_category}' category currently leads with total sales of {format_brl_inr(top_category_sales_brl)}.")
    if significant_changes:
        alerts.append("Significant changes detected for some categories.")
    if new_entries:
        alerts.append(f"New entries to Top 5: {', '.join(new_entries)}")
    if dropped:
        alerts.append(f"Dropped from Top 5: {', '.join(dropped)}")

    # Log & snapshot
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = {
        "Timestamp": timestamp,
        "Alert_Message": " | ".join(alerts),
        "Top_5_Categories": ", ".join(current_top_5),
        "Top_Category": top_category,
        "Top_Category_Sales": top_category_sales_brl,
        "Avg_Review_Score": avg_review,
        "Total_Sales_BRL": total_sales_brl,
        "Avg_Delivery_Delay": avg_delay
    }
    log_df = pd.DataFrame([log_entry]) if log_df_prev.empty else pd.concat([log_df_prev, pd.DataFrame([log_entry])], ignore_index=True)
    snapshot_df = pd.DataFrame({
        "Category": list(category_sales.index.astype(str)),
        "Sales_BRL": list(category_sales.values.astype(float))
    }).sort_values("Sales_BRL", ascending=False)
    write_alerts_sheets(ALERTS_PATH, log_df, snapshot_df)

    # ----------------- ARIMA FORECAST (monthly) -----------------
    forecast = None
    future_dates = None
    forecast_df = pd.DataFrame()
    model_order = None

    # ensure timestamp col exists
    if "order_purchase_timestamp" not in df.columns:
        print("❌ 'order_purchase_timestamp' column missing — skipping ARIMA.")
    else:
        df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
        # safe monthly grouping: produce column named 'month' (Period -> Timestamp EOM)
        monthly_sales = (
            df.dropna(subset=["order_purchase_timestamp"])
              .groupby(df["order_purchase_timestamp"].dt.to_period("M").rename("month"))[PRICE_COL]
              .sum()
              .reset_index()
        )
        # convert period to timestamp (month end)
        if pd.api.types.is_period_dtype(monthly_sales["month"].dtype):
            monthly_sales["month"] = monthly_sales["month"].dt.to_timestamp("M")
        else:
            try:
                monthly_sales["month"] = pd.to_datetime(monthly_sales["month"])
            except Exception:
                pass

        # Apply threshold filter
        monthly_sales = monthly_sales[monthly_sales[PRICE_COL] > 50000]

        if monthly_sales.empty:
            print("❌ Not enough data for ARIMA forecast after filtering (>50,000).")
        else:
            ts = monthly_sales.set_index("month")[PRICE_COL]
            try:
                model = auto_arima(ts, seasonal=False, trace=False, error_action="ignore", suppress_warnings=True)
                n_periods = 12
                pred_forecast, conf_int = model.predict(n_periods=n_periods, return_conf_int=True)
                forecast = np.asarray(pred_forecast)
                conf_int = np.asarray(conf_int)
                # future monthly dates (end of month)
                future_dates = pd.date_range(start=ts.index[-1], periods=n_periods+1, freq="M")[1:]
                forecast_df = pd.DataFrame({
                    "ds": future_dates,
                    "forecast_sales": forecast,
                    "lower_bound": conf_int[:, 0],
                    "upper_bound": conf_int[:, 1]
                })
                # Plot and save
                plt.figure(figsize=(12, 6))
                plt.plot(ts.index, ts.values, label="Historical Sales")
                plt.plot(future_dates, forecast, label="Forecast")
                plt.fill_between(future_dates, conf_int[:, 0], conf_int[:, 1], alpha=0.2)
                plt.title("Monthly Sales Forecast (AutoARIMA)")
                plt.xlabel("Date")
                plt.ylabel("Payment Value (Sales)")
                plt.legend()
                plt.grid(True)
                plt.savefig(ARIMA_PLOT_PATH)
                plt.close()
                model_order = getattr(model, "order", None)
            except Exception as e:
                print("❌ ARIMA modeling failed:", e)
                traceback.print_exc()
                forecast = None
                future_dates = None
                forecast_df = pd.DataFrame()
                model_order = None

    # ----------------- PDF -----------------
    comparison_sorted = sorted(comparison_data, key=lambda x: abs(x[2]), reverse=True)
    comparison_for_pdf = comparison_sorted[:40]
    generate_pdf_report(PDF_REPORT_PATH, ARIMA_PLOT_PATH, timestamp, alerts, current_top_5, top_category,
                        avg_review, total_sales_fmt, avg_delay_fmt, comparison_for_pdf, new_entries, dropped,
                        arima_order=model_order, forecast_df=forecast_df)

    # ----------------- EMAIL: greeting + one alert line + ARIMA sentence -----------------
    greeting = "Hello,\nWelcome to QuickStats."
    one_alert_line = alerts[0] if alerts else "No alerts."

    if forecast is not None and hasattr(forecast, "__len__") and len(forecast) >= 1:
        try:
            first_val = float(forecast[0])
            last_val = float(forecast[-1])
            arima_sentence = f"Expected future sales rise from {first_val:,.0f} to {last_val:,.0f} within 12 months."
        except Exception:
            arima_sentence = "Forecast unavailable"
    else:
        arima_sentence = "Forecast unavailable"

    body_lines = [
        greeting,
        "",
        one_alert_line,
        arima_sentence
    ]
    email_body = "\n".join(body_lines)

    send_email("QuickStats: Alert + ARIMA Summary", email_body, attachment_path=PDF_REPORT_PATH)

# ----------------- RUN -----------------
if __name__ == "__main__":
    main()


🔁 Loading data from: /content/drive/MyDrive/Major/olist_order_dataset_cleaned.xlsx
✅ Alerts file updated with 'log' and 'snapshot' sheets: /content/drive/MyDrive/Major/Alerts_Data.xlsx


/tmp/ipython-input-92534638.py:342: DeprecationWarning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(monthly_sales["month"].dtype):
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/tmp/ipython-input-92534638.py:364: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  future_dates = pd.date_range(start=ts.index[-1], periods=n_periods+1, freq="M")[1:]


✅ PDF report generated with ARIMA graph and findings: /content/drive/MyDrive/Major/QuickStats_ARIMA_Report.pdf
📧 Email alert sent successfully from aurevella234@gmail.com


Import Libraries


In [ ]:
# full_quickstats_arima_combined_with_arima_details_email_short_final.py
# Combines QuickStats alerts + Auto-ARIMA forecasting
# PDF report contains both alerts and ARIMA graph + ARIMA findings (findings below graph)
# Email contains greeting, one alert line, and the ARIMA summary sentence

import os
import pandas as pd
import smtplib
import traceback
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
import matplotlib.pyplot as plt
import numpy as np

# PDF generation imports
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont

# ARIMA import
from pmdarima import auto_arima


In [ ]:
# ----------------- CONFIG -----------------
DATA_PATH = "/content/drive/MyDrive/Major/olist_order_dataset_cleaned.xlsx"
ALERTS_PATH = "/content/drive/MyDrive/Major/Alerts_Data.xlsx"
PDF_REPORT_PATH = "/content/drive/MyDrive/Major/QuickStats_ARIMA_Report.pdf"
ARIMA_PLOT_PATH = "/content/drive/MyDrive/Major/QuickStats_ARIMA_Plot.png"

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

SENDERS = [
    {"email": "aurevella234@gmail.com", "password": "etkk dmew ocmw tdvd"}
]

RECEIVER_EMAILS = ["sapna.bhola86@gmail.com"]

CATEGORY_COL = "general_category"
PRICE_COL = "payment_value"
REVIEW_COL = "review_score"
DELAY_COL = "delivery_delay_days"

BRL_TO_INR = 16.5

# ----------------- UTILS -----------------
def safe_read_excel(path):
    try:
        return pd.read_excel(path)
    except Exception as e:
        print(f"❌ Error reading {path}: {e}")
        return pd.DataFrame()

def read_alerts_sheets(path):
    if not os.path.exists(path):
        return pd.DataFrame(), pd.DataFrame()
    try:
        all_sheets = pd.read_excel(path, sheet_name=None)
        return all_sheets.get("log", pd.DataFrame()), all_sheets.get("snapshot", pd.DataFrame())
    except Exception as e:
        print(f"❌ Failed to read sheets from {path}: {e}")
        return pd.DataFrame(), pd.DataFrame()

def write_alerts_sheets(path, log_df, snapshot_df):
    try:
        with pd.ExcelWriter(path, engine="openpyxl", mode="w") as writer:
            log_df.to_excel(writer, sheet_name="log", index=False)
            snapshot_df.to_excel(writer, sheet_name="snapshot", index=False)
        print("✅ Alerts file updated with 'log' and 'snapshot' sheets:", path)
    except Exception as e:
        print("❌ Failed to write alerts sheets:", e)
        traceback.print_exc()

def format_brl_inr(value_brl):
    try:
        value_inr = value_brl * BRL_TO_INR
    except Exception:
        value_inr = 0
    return f"R${value_brl:,.2f} (≈ ₹{value_inr:,.0f})"

def format_delay(value):
    return "No delay (early deliveries)" if value < 0 else f"{value:.1f} days"

In [ ]:
# ----------------- EMAIL -----------------
def send_email(subject, body, attachment_path=None):
    sender = SENDERS[0]
    try:
        msg = MIMEMultipart()
        msg["From"] = sender["email"]
        msg["To"] = ", ".join(RECEIVER_EMAILS)
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        if attachment_path and os.path.exists(attachment_path):
            with open(attachment_path, "rb") as f:
                part = MIMEApplication(f.read(), Name=os.path.basename(attachment_path))
            part["Content-Disposition"] = f'attachment; filename="{os.path.basename(attachment_path)}"'
            msg.attach(part)

        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.starttls()
            server.login(sender["email"], sender["password"])
            server.send_message(msg)
        print(f"📧 Email alert sent successfully from {sender['email']}")
    except Exception as e:
        print(f"❌ Failed to send email from {sender['email']}: {e}")
        traceback.print_exc()

In [ ]:
# ----------------- PDF GENERATION -----------------
def generate_pdf_report(pdf_path, plot_path, timestamp, alerts, current_top_5, top_category,
                        avg_review, total_sales_fmt, avg_delay_fmt, comparison_data, new_entries, dropped,
                        arima_order=None, forecast_df=None):
    try:
        pdfmetrics.registerFont(UnicodeCIDFont("HeiseiMin-W3"))
    except Exception:
        pass

    doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=40, leftMargin=40, topMargin=60, bottomMargin=40)
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "TitleStyle", parent=styles["Title"], alignment=TA_CENTER,
        fontName="HeiseiMin-W3", fontSize=18, textColor=colors.HexColor("#1F4E79"), spaceAfter=20
    )
    section_header = ParagraphStyle(
        "SectionHeader", parent=styles["Heading2"],
        fontName="HeiseiMin-W3", fontSize=12, textColor=colors.HexColor("#0A2540"),
        spaceBefore=10, spaceAfter=6
    )
    normal_text = ParagraphStyle("NormalText", parent=styles["Normal"], fontName="HeiseiMin-W3", fontSize=10, leading=13)
    footer_style = ParagraphStyle("Footer", parent=styles["Normal"], alignment=TA_CENTER,
                                  fontName="HeiseiMin-W3", fontSize=9, textColor=colors.grey)

    content = []
    content.append(Paragraph("QuickStats Performance Summary Report", title_style))
    content.append(Paragraph(f"<b>Date:</b> {timestamp}", normal_text))
    content.append(Spacer(1, 10))

    # Alerts
    content.append(Paragraph("📢 Alert Summary", section_header))
    for alert in alerts:
        content.append(Paragraph(f"• {alert}", normal_text))
    content.append(Spacer(1, 10))

    # Metrics Table
    table_data = [
        ["Top 5 Categories", Paragraph(", ".join(current_top_5), normal_text)],
        ["Top Category by Sales", Paragraph(top_category, normal_text)],
        ["Average Review Score", Paragraph(str(avg_review), normal_text)],
        ["Total Sales", Paragraph(total_sales_fmt, normal_text)],
        ["Average Delivery Delay", Paragraph(avg_delay_fmt, normal_text)]
    ]
    table = Table(table_data, colWidths=[160, 340])
    table.setStyle(TableStyle([
        ("FONTNAME", (0, 0), (-1, -1), "HeiseiMin-W3"),
        ("ALIGN", (0, 0), (-1, -1), "LEFT"),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("INNERGRID", (0, 0), (-1, -1), 0.25, colors.grey),
        ("BOX", (0, 0), (-1, -1), 0.25, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.HexColor("#F7F9FB")])
    ]))
    content.append(table)
    content.append(Spacer(1, 12))


In [ ]:
 # Category Comparison
    content.append(Paragraph("📈 Category Sales Comparison (vs Previous Report)", section_header))
    if new_entries:
        content.append(Paragraph(f"🆕 New categories entered Top 5: {', '.join(new_entries)}", normal_text))
    if dropped:
        content.append(Paragraph(f"⬇ Categories dropped from Top 5: {', '.join(dropped)}", normal_text))
    content.append(Spacer(1, 6))
    if comparison_data:
        comp_table_data = [["Category", "Previous Sales (BRL)", "Added Sales (BRL)", "Current Sales (BRL)"]]
        for cat, prev_sales, added_sales, cur_sales in comparison_data:
            comp_table_data.append([cat, f"{prev_sales:,.2f}", f"{added_sales:,.2f}", f"{cur_sales:,.2f}"])
        t = Table(comp_table_data, colWidths=[180, 110, 110, 110])
        t.setStyle(TableStyle([
            ("FONTNAME", (0, 0), (-1, -1), "HeiseiMin-W3"),
            ("FONTSIZE", (0, 0), (-1, -1), 9),
            ("GRID", (0, 0), (-1, -1), 0.25, colors.grey),
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9E1F2")),
            ("ALIGN", (1, 1), (-1, -1), "RIGHT")
        ]))
        content.append(t)
    content.append(Spacer(1, 15))

    # ARIMA Graph and findings ON THE SAME PAGE (graph starts on new page)
    content.append(PageBreak())
    content.append(Paragraph("📈 ARIMA Forecast Graph", section_header))
    if plot_path and os.path.exists(plot_path):
        content.append(Image(plot_path, width=500, height=300))
    content.append(Spacer(1, 12))

    # Findings directly below the graph (same page)
    content.append(Paragraph("📊 Sales Forecast Findings", section_header))
    if arima_order and forecast_df is not None and not forecast_df.empty:
        last_actual = float(forecast_df['forecast_sales'].iloc[0])
        future_min = float(forecast_df['lower_bound'].min())
        future_max = float(forecast_df['upper_bound'].max())
        future_last = float(forecast_df['forecast_sales'].iloc[-1])
        trend = "positive and upward" if future_last > last_actual else "flat/slightly downward"

        findings_text = (
            f"Historical sales show strong growth in recent periods.\n"
            f"Expected future sales rise from {last_actual:,.0f} to {future_last:,.0f} within {len(forecast_df)} months.\n"
            f"The confidence interval suggests sales may vary between {future_min:,.0f} and {future_max:,.0f}, showing some uncertainty.\n"
            f"Overall forecast trend = {trend}."
        )
        for line in findings_text.split("\n"):
            content.append(Paragraph(line, normal_text))
    else:
        content.append(Paragraph("ARIMA forecast details are unavailable (not enough data).", normal_text))

    content.append(Spacer(1, 20))
    content.append(Paragraph("Confidential | Auto-generated by QuickStats Monitoring System", footer_style))

    doc.build(content)
    print(f"✅ PDF report generated with ARIMA graph and findings: {pdf_path}")

# ----------------- MAIN -----------------
def main():
    print("🔁 Loading data from:", DATA_PATH)
    df = safe_read_excel(DATA_PATH)
    if df.empty:
        print("❌ No data loaded. Aborting alert run.")
        return

    # Ensure numeric columns exist and are numeric
    df[PRICE_COL] = pd.to_numeric(df.get(PRICE_COL, pd.Series(dtype=float)), errors="coerce").fillna(0)
    df[REVIEW_COL] = pd.to_numeric(df.get(REVIEW_COL, pd.Series(dtype=float)), errors="coerce").fillna(0)
    df[DELAY_COL] = pd.to_numeric(df.get(DELAY_COL, pd.Series(dtype=float)), errors="coerce").fillna(0)

    # QuickStats Metrics
    category_sales = df.groupby(CATEGORY_COL)[PRICE_COL].sum().sort_values(ascending=False)
    current_top_5 = list(category_sales.head(5).index.astype(str))
    top_category = current_top_5[0] if current_top_5 else ""
    top_category_sales_brl = float(category_sales.iloc[0]) if not category_sales.empty else 0.0
    avg_review = round(df[REVIEW_COL].mean(), 2) if REVIEW_COL in df.columns else 0.0
    total_sales_brl = float(round(df[PRICE_COL].sum(), 2))
    avg_delay = round(df[DELAY_COL].mean(), 2) if DELAY_COL in df.columns else 0.0
    total_sales_fmt = format_brl_inr(total_sales_brl)
    avg_delay_fmt = format_delay(avg_delay)

    # Previous alerts loading
    log_df_prev, snapshot_prev = read_alerts_sheets(ALERTS_PATH)
    prev_top_5 = []
    prev_sales_map = {}
    if not snapshot_prev.empty and "Category" in snapshot_prev.columns and "Sales_BRL" in snapshot_prev.columns:
        prev_snapshot_sorted = snapshot_prev.sort_values("Sales_BRL", ascending=False)
        prev_top_5 = list(prev_snapshot_sorted.head(5)["Category"].astype(str))
        prev_sales_map = dict(zip(snapshot_prev["Category"].astype(str), snapshot_prev["Sales_BRL"].astype(float)))

    # Comparison & alerts
    all_categories = set(category_sales.index.astype(str)).union(set(prev_sales_map.keys()))
    comparison_data = []
    significant_changes = []
    for cat in sorted(all_categories):
        cur = float(category_sales.get(cat, 0.0)) if cat in category_sales.index.astype(str) else 0.0
        prev = float(prev_sales_map.get(cat, 0.0))
        added = cur - prev
        comparison_data.append((cat, prev, added, cur))
        if abs(added) >= 10.0:
            trend = "increased" if added > 0 else "decreased"
            significant_changes.append(f"'{cat}' has {trend} by R${abs(added):,.2f} (from R${prev:,.2f} to R${cur:,.2f}).")

    new_entries = [c for c in current_top_5 if c not in prev_top_5]
    dropped = [c for c in prev_top_5 if c not in current_top_5]

    alerts = []
    prev_top_cat = prev_top_5[0] if prev_top_5 else None
    if prev_top_cat and prev_top_cat != top_category:
        alerts.append(f"Top category has changed from '{prev_top_cat}' to '{top_category}'.")
    elif top_category:
        alerts.append(f"Top category remains '{top_category}'.")
    prev_top_sales = prev_sales_map.get(top_category, None)
    if prev_top_sales is not None and prev_top_sales > 0:
        diff_top = top_category_sales_brl - prev_top_sales
        if abs(diff_top) >= 10.0:
            trend = "increased" if diff_top > 0 else "decreased"
            alerts.append(f"Sales for top category '{top_category}' have {trend} by R${abs(diff_top):,.2f} since last snapshot.")
    alerts.append(f"'{top_category}' category currently leads with total sales of {format_brl_inr(top_category_sales_brl)}.")
    if significant_changes:
        alerts.append("Significant changes detected for some categories.")
    if new_entries:
        alerts.append(f"New entries to Top 5: {', '.join(new_entries)}")
    if dropped:
        alerts.append(f"Dropped from Top 5: {', '.join(dropped)}")

    # Log & snapshot
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = {
        "Timestamp": timestamp,
        "Alert_Message": " | ".join(alerts),
        "Top_5_Categories": ", ".join(current_top_5),
        "Top_Category": top_category,
        "Top_Category_Sales": top_category_sales_brl,
        "Avg_Review_Score": avg_review,
        "Total_Sales_BRL": total_sales_brl,
        "Avg_Delivery_Delay": avg_delay
    }
    log_df = pd.DataFrame([log_entry]) if log_df_prev.empty else pd.concat([log_df_prev, pd.DataFrame([log_entry])], ignore_index=True)
    snapshot_df = pd.DataFrame({
        "Category": list(category_sales.index.astype(str)),
        "Sales_BRL": list(category_sales.values.astype(float))
    }).sort_values("Sales_BRL", ascending=False)
    write_alerts_sheets(ALERTS_PATH, log_df, snapshot_df)

In [ ]:
# ----------------- ARIMA FORECAST (monthly) -----------------
    forecast = None
    future_dates = None
    forecast_df = pd.DataFrame()
    model_order = None

    # ensure timestamp col exists
    if "order_purchase_timestamp" not in df.columns:
        print("❌ 'order_purchase_timestamp' column missing — skipping ARIMA.")
    else:
        df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
        # safe monthly grouping: produce column named 'month' (Period -> Timestamp EOM)
        monthly_sales = (
            df.dropna(subset=["order_purchase_timestamp"])
              .groupby(df["order_purchase_timestamp"].dt.to_period("M").rename("month"))[PRICE_COL]
              .sum()
              .reset_index()
        )
        # convert period to timestamp (month end)
        if pd.api.types.is_period_dtype(monthly_sales["month"].dtype):
            monthly_sales["month"] = monthly_sales["month"].dt.to_timestamp("M")
        else:
            try:
                monthly_sales["month"] = pd.to_datetime(monthly_sales["month"])
            except Exception:
                pass

        # Apply threshold filter
        monthly_sales = monthly_sales[monthly_sales[PRICE_COL] > 50000]

        if monthly_sales.empty:
            print("❌ Not enough data for ARIMA forecast after filtering (>50,000).")
        else:
            ts = monthly_sales.set_index("month")[PRICE_COL]
            try:
                model = auto_arima(ts, seasonal=False, trace=False, error_action="ignore", suppress_warnings=True)
                n_periods = 12
                pred_forecast, conf_int = model.predict(n_periods=n_periods, return_conf_int=True)
                forecast = np.asarray(pred_forecast)
                conf_int = np.asarray(conf_int)
                # future monthly dates (end of month)
                future_dates = pd.date_range(start=ts.index[-1], periods=n_periods+1, freq="M")[1:]
                forecast_df = pd.DataFrame({
                    "ds": future_dates,
                    "forecast_sales": forecast,
                    "lower_bound": conf_int[:, 0],
                    "upper_bound": conf_int[:, 1]
                })
                # Plot and save
                plt.figure(figsize=(12, 6))
                plt.plot(ts.index, ts.values, label="Historical Sales")
                plt.plot(future_dates, forecast, label="Forecast")
                plt.fill_between(future_dates, conf_int[:, 0], conf_int[:, 1], alpha=0.2)
                plt.title("Monthly Sales Forecast (AutoARIMA)")
                plt.xlabel("Date")
                plt.ylabel("Payment Value (Sales)")
                plt.legend()
                plt.grid(True)
                plt.savefig(ARIMA_PLOT_PATH)
                plt.close()
                model_order = getattr(model, "order", None)
            except Exception as e:
                print("❌ ARIMA modeling failed:", e)
                traceback.print_exc()
                forecast = None
                future_dates = None
                forecast_df = pd.DataFrame()
                model_order = None

    # ----------------- PDF -----------------
    comparison_sorted = sorted(comparison_data, key=lambda x: abs(x[2]), reverse=True)
    comparison_for_pdf = comparison_sorted[:40]
    generate_pdf_report(PDF_REPORT_PATH, ARIMA_PLOT_PATH, timestamp, alerts, current_top_5, top_category,
                        avg_review, total_sales_fmt, avg_delay_fmt, comparison_for_pdf, new_entries, dropped,
                        arima_order=model_order, forecast_df=forecast_df)


In [ ]:

    # ----------------- EMAIL: greeting + one alert line + ARIMA sentence -----------------
    greeting = "Hello,\nWelcome to QuickStats."
    one_alert_line = alerts[0] if alerts else "No alerts."

    if forecast is not None and hasattr(forecast, "__len__") and len(forecast) >= 1:
        try:
            first_val = float(forecast[0])
            last_val = float(forecast[-1])
            arima_sentence = f"Expected future sales rise from {first_val:,.0f} to {last_val:,.0f} within 12 months."
        except Exception:
            arima_sentence = "Forecast unavailable"
    else:
        arima_sentence = "Forecast unavailable"

    body_lines = [
        greeting,
        "",
        one_alert_line,
        arima_sentence
    ]
    email_body = "\n".join(body_lines)

    send_email("QuickStats: Alert + ARIMA Summary", email_body, attachment_path=PDF_REPORT_PATH)

# ----------------- RUN -----------------
if __name__ == "__main__":
    main()